In [1]:
import sys
sys.path.append("..")

import librosa
import soundfile as sf
import numpy as np
import pandas as pd

df = pd.read_csv('../data/track_analysis_results.csv')
print(f"Loaded {len(df)} tracks")
df.head()

Loaded 50 tracks


,filename,genre,bpm,key,camelot,energy,key_confidence
0,../data/genres_original\classical\classical.00...,classical,95.70,D Major,10B,0.098,0.81
1,../data/genres_original\classical\classical.00...,classical,112.35,B Minor,10A,0.079,0.92
2,../data/genres_original\classical\classical.00...,classical,99.38,E Major,12B,0.107,0.70
3,../data/genres_original\classical\classical.00...,classical,71.78,C Major,8B,0.088,0.75
4,../data/genres_original\classical\classical.00...,classical,161.50,A Minor,8A,0.115,0.82


In [2]:
from src.analyzer import compatibility_score, greedy_reorder, two_opt_improve

ordered = greedy_reorder(df, lookahead=3)
ordered_list = ordered.to_dict('records')
final_playlist, final_score, _ = two_opt_improve(ordered_list)

print(f"Final playlist score: {final_score:.2f}/100")

Final playlist score: 74.95/100


In [12]:
def make_transition_clip(file_a, file_b, clip_seconds=8, crossfade_seconds=3, sr=22050):
    """
    Extracts the tail of file_a and head of file_b, and crossfades them
    so the transition sounds natural instead of a hard cut.
    """
    y_a, _ = librosa.load(file_a, sr=sr)
    y_b, _ = librosa.load(file_b, sr=sr)
    
    n_clip = clip_seconds * sr
    n_fade = crossfade_seconds * sr
    
    end_of_a   = y_a[-n_clip:] if len(y_a) > n_clip else y_a
    start_of_b = y_b[:n_clip] if len(y_b) > n_clip else y_b
    
    # Build fade curves — linear ramp down for A, ramp up for B
    fade_out = np.linspace(1.0, 0.0, n_fade)
    fade_in  = np.linspace(0.0, 1.0, n_fade)
    
    # Apply fade to the last n_fade samples of A and first n_fade samples of B
    end_of_a_faded = end_of_a.copy()
    end_of_a_faded[-n_fade:] *= fade_out
    
    start_of_b_faded = start_of_b.copy()
    start_of_b_faded[:n_fade] *= fade_in
    
    # The overlapping region: add the faded tail of A to the faded head of B
    overlap = end_of_a_faded[-n_fade:] + start_of_b_faded[:n_fade]
    
    # Stitch: [non-overlapping part of A] + [overlap] + [non-overlapping part of B]
    transition = np.concatenate([
        end_of_a_faded[:-n_fade],
        overlap,
        start_of_b_faded[n_fade:]
    ])
    
    return transition, sr

In [13]:
# Pick the worst transition from ORIGINAL random order (before any reordering)
# vs the best transition from FINAL playlist

# Best transition example — first two tracks in the final playlist
song_a = final_playlist[0]['filename']
song_b = final_playlist[1]['filename']

clip, sr = make_transition_clip(song_a, song_b, clip_seconds=8)
sf.write("good_transition.wav", clip, sr)

print(f"Good transition: {song_a} → {song_b}")
print(f"Compatibility score: {compatibility_score(final_playlist[0], final_playlist[1])}")
print("Saved: good_transition.wav")

Good transition: ../data/genres_original\classical\classical.00009.wav → ../data/genres_original\classical\classical.00007.wav
Compatibility score: 58.6
Saved: good_transition.wav


In [15]:
# Find the actual best and worst transitions in the final playlist
pair_scores = []
for i in range(len(final_playlist) - 1):
    score = compatibility_score(final_playlist[i], final_playlist[i+1])
    pair_scores.append((score, i))

pair_scores.sort(reverse=True)

best_score, best_i = pair_scores[0]
worst_score, worst_i = pair_scores[-1]

print(f"Best transition in final playlist:  index {best_i} → {best_i+1}, score {best_score}")
print(f"  {final_playlist[best_i]['filename']} → {final_playlist[best_i+1]['filename']}")

print(f"\nWorst transition in final playlist: index {worst_i} → {worst_i+1}, score {worst_score}")
print(f"  {final_playlist[worst_i]['filename']} → {final_playlist[worst_i+1]['filename']}")

Best transition in final playlist:  index 22 → 23, score 95.9
  ../data/genres_original\hiphop\hiphop.00004.wav → ../data/genres_original\rock\rock.00005.wav

Worst transition in final playlist: index 5 → 6, score 45.3
  ../data/genres_original\classical\classical.00005.wav → ../data/genres_original\rock\rock.00006.wav


In [16]:
song_a = final_playlist[best_i]['filename']
song_b = final_playlist[best_i + 1]['filename']

clip, sr = make_transition_clip(song_a, song_b, clip_seconds=8)
sf.write("good_simple_transition.wav", clip, sr)

print(f"Good transition: {song_a} → {song_b}, score: {best_score}")

Good transition: ../data/genres_original\hiphop\hiphop.00004.wav → ../data/genres_original\rock\rock.00005.wav, score: 95.9


In [17]:
song_c = final_playlist[worst_i]['filename']
song_d = final_playlist[worst_i + 1]['filename']

clip, sr = make_transition_clip(song_c, song_d, clip_seconds=8)
sf.write("bad_simple_transition.wav", clip, sr)

print(f"Bad transition: {song_c} → {song_d}, score: {worst_score}")

Bad transition: ../data/genres_original\classical\classical.00005.wav → ../data/genres_original\rock\rock.00006.wav, score: 45.3


In [18]:
# Best pair
a = final_playlist[best_i]
b = final_playlist[best_i + 1]
print("BEST PAIR (score 95.9)")
print(f"  A: {a['filename']} | BPM {a['bpm']} | {a['camelot']} | Energy {a['energy']}")
print(f"  B: {b['filename']} | BPM {b['bpm']} | {b['camelot']} | Energy {b['energy']}")

# Worst pair
c = final_playlist[worst_i]
d = final_playlist[worst_i + 1]
print("\nWORST PAIR (score 45.3)")
print(f"  C: {c['filename']} | BPM {c['bpm']} | {c['camelot']} | Energy {c['energy']}")
print(f"  D: {d['filename']} | BPM {d['bpm']} | {d['camelot']} | Energy {d['energy']}")

BEST PAIR (score 95.9)
  A: ../data/genres_original\hiphop\hiphop.00004.wav | BPM 95.7 | 8A | Energy 0.351
  B: ../data/genres_original\rock\rock.00005.wav | BPM 99.38 | 8A | Energy 0.269

WORST PAIR (score 45.3)
  C: ../data/genres_original\classical\classical.00005.wav | BPM 143.55 | 5B | Energy 0.103
  D: ../data/genres_original\rock\rock.00006.wav | BPM 112.35 | 4B | Energy 0.257


In [20]:
import os

os.makedirs("transitions", exist_ok=True)

transition_files = []

for i in range(len(final_playlist) - 1):
    song_a = final_playlist[i]['filename']
    song_b = final_playlist[i + 1]['filename']
    score  = compatibility_score(final_playlist[i], final_playlist[i + 1])
    
    clip, sr = make_transition_clip(song_a, song_b, clip_seconds=10, crossfade_seconds=3)
    
    out_path = f"transitions/transition_{i:02d}_{i+1:02d}.wav"
    sf.write(out_path, clip, sr)
    
    transition_files.append({
        "index": i,
        "song_a": song_a,
        "song_b": song_b,
        "score": score,
        "path": out_path
    })
    print(f"  {i:2d}→{i+1:2d}  score: {score:5.1f}  saved: {out_path}")

print(f"\nGenerated {len(transition_files)} transition clips.")

   0→ 1  score:  58.6  saved: transitions/transition_00_01.wav
   1→ 2  score:  52.9  saved: transitions/transition_01_02.wav
   2→ 3  score:  86.7  saved: transitions/transition_02_03.wav
   3→ 4  score:  86.2  saved: transitions/transition_03_04.wav
   4→ 5  score:  73.2  saved: transitions/transition_04_05.wav
   5→ 6  score:  45.3  saved: transitions/transition_05_06.wav
   6→ 7  score:  78.3  saved: transitions/transition_06_07.wav
   7→ 8  score:  54.6  saved: transitions/transition_07_08.wav
   8→ 9  score:  49.4  saved: transitions/transition_08_09.wav
   9→10  score:  57.9  saved: transitions/transition_09_10.wav
  10→11  score:  90.7  saved: transitions/transition_10_11.wav
  11→12  score:  60.9  saved: transitions/transition_11_12.wav
  12→13  score:  71.2  saved: transitions/transition_12_13.wav
  13→14  score:  83.0  saved: transitions/transition_13_14.wav
  14→15  score:  86.2  saved: transitions/transition_14_15.wav
  15→16  score:  64.8  saved: transitions/transition_15